# 01 - Data Preprocessing

This notebook loads the original CICIDS2017 dataset and performs one preprocessing task at a time. It creates one processed file for model training.

## Step 1 - Import the libraries

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

## Step 2 - Set the file paths

In [ ]:
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_FILE = PROJECT / 'data' / 'CICIDS2017_combined.csv'
PROCESSED_FILE = PROJECT / 'data' / 'processed_data.npz'
MODELS_DIR = PROJECT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

## Step 3 - Load the original dataset

Keep `ROWS_TO_LOAD = None` for the complete assignment dataset. The complete CSV is large, so close unnecessary applications before running this cell.

In [ ]:
ROWS_TO_LOAD = None

if not DATA_FILE.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_FILE}')

df = pd.read_csv(DATA_FILE, nrows=ROWS_TO_LOAD, low_memory=False)
original_rows = len(df)
print('Dataset shape:', df.shape)
display(df.head())

## Step 4 - Clean the column names

In [ ]:
df.columns = [str(column).strip() for column in df.columns]
print('Column names cleaned.')

## Step 5 - Remove duplicated columns

In [ ]:
duplicate_columns = [
    column for column in df.columns
    if column.endswith('.1') and column[:-2] in df.columns
]
df = df.drop(columns=duplicate_columns)
print('Duplicated columns removed:', duplicate_columns)

## Step 6 - Remove duplicated rows

In [ ]:
columns_used_for_duplicates = [column for column in df.columns if column != '_source_file']
rows_before_duplicates = len(df)
df = df.drop_duplicates(subset=columns_used_for_duplicates).reset_index(drop=True)
duplicates_removed = rows_before_duplicates - len(df)
print('Duplicated rows removed:', duplicates_removed)

## Step 7 - Clean and group the labels

In [ ]:
df['Label'] = df['Label'].astype(str).str.strip()

def group_label(label):
    label = label.replace('–', '-').replace('�', '-')
    if label == 'BENIGN':
        return 'BENIGN'
    if label == 'Bot':
        return 'Botnet'
    if label == 'DDoS':
        return 'DDoS'
    if label.startswith('DoS'):
        return 'DoS'
    if label.startswith('Web Attack'):
        return 'Web Attack'
    if 'Patator' in label:
        return 'Brute Force'
    if label in ['PortScan', 'Infiltration', 'Heartbleed']:
        return label
    return None

df['attack_type'] = df['Label'].map(group_label)
unknown_labels = df.loc[df['attack_type'].isna(), 'Label'].unique()
if len(unknown_labels):
    raise ValueError(f'Unknown labels: {unknown_labels}')

display(df['attack_type'].value_counts().to_frame('rows'))

## Step 8 - Separate the features and target

In [ ]:
columns_not_used_for_training = ['Label', 'attack_type', '_source_file']
feature_columns = [
    column for column in df.columns
    if column not in columns_not_used_for_training
]
X = df[feature_columns].copy()
y = df['attack_type'].copy()
print('Feature shape:', X.shape)
print('Target shape:', y.shape)

## Step 9 - Convert the features to numeric values

In [ ]:
X = X.apply(pd.to_numeric, errors='coerce')
print('All features converted to numeric values.')

## Step 10 - Replace infinity values

In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)
empty_columns = X.columns[X.isna().all()].tolist()
X = X.drop(columns=empty_columns)
feature_columns = X.columns.tolist()
print('Missing values to fill:', int(X.isna().sum().sum()))

## Step 11 - Split the data into training and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))

## Step 12 - Create the feature preprocessor

In [ ]:
preprocessor = Pipeline([
    ('fill_missing_values', SimpleImputer(strategy='median')),
    ('remove_constant_features', VarianceThreshold()),
    ('standardize_features', StandardScaler()),
])
preprocessor

## Step 13 - Fit the preprocessor using only the training data

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train).astype(np.float32)
print('Processed training shape:', X_train_processed.shape)

## Step 14 - Apply the same preprocessing to the test data

In [ ]:
X_test_processed = preprocessor.transform(X_test).astype(np.float32)
print('Processed test shape:', X_test_processed.shape)

## Step 15 - Encode the class labels

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train).astype(np.int64)
y_test_encoded = label_encoder.transform(y_test).astype(np.int64)
print('Encoded classes:', list(label_encoder.classes_))

## Step 16 - Save one processed dataset

In [ ]:
np.savez_compressed(
    PROCESSED_FILE,
    X_train=X_train_processed,
    X_test=X_test_processed,
    y_train=y_train_encoded,
    y_test=y_test_encoded,
)
print('Saved:', PROCESSED_FILE)

## Step 17 - Save the preprocessing objects

In [ ]:
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.joblib')
joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.joblib')
print('Preprocessing objects saved in the models folder.')

## Step 18 - Check the final output

In [ ]:
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()
assert len(X_train_processed) == len(y_train_encoded)
assert len(X_test_processed) == len(y_test_encoded)

print('Original rows:', original_rows)
print('Cleaned rows:', len(df))
print('Duplicates removed:', duplicates_removed)
print('Number of features:', X_train_processed.shape[1])
print('Preprocessing completed successfully.')